In [2]:
import pandas as pd
import random

df = pd.read_pickle(r"C:\Users\Eman\Healthcare-RAG-Powered-Medical-QA-Assistant\data\chunk_mapping.pkl")

# must contain: question, answer, text_chunk
print(df.columns)

Index(['chunk_id', 'question', 'answer', 'text_chunk'], dtype='object')


In [3]:
random.seed(42)

sampled_df = df.sample(n=200, random_state=42)

questions = sampled_df["question"].tolist()
references = sampled_df["answer"].tolist()

In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.rag.pipeline import rag_pipeline
from transformers import pipeline

c:\Users\Eman\Healthcare-RAG-Powered-Medical-QA-Assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
plain_llm = pipeline(
    "text-generation",
    model="distilgpt2"
)

# IMPORTANT FIX (removes warnings)
plain_llm.tokenizer.pad_token = plain_llm.tokenizer.eos_token

In [10]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [6]:
rag_outputs = []
llm_outputs = []

for q in questions:
    # RAG system
    rag_outputs.append(rag_pipeline(q))

    # baseline LLM
    llm_outputs.append(
        plain_llm(q, max_new_tokens=100)[0]["generated_text"]
    )

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

In [7]:
def clean(text):
    return text.replace("Answer:", "").strip()

rag_outputs = [clean(x) for x in rag_outputs]
llm_outputs = [clean(x) for x in llm_outputs]

In [8]:
from src.evaluation.metrics import compute_bleu, compute_rouge

rag_bleu = compute_bleu(rag_outputs, references)
rag_rouge = compute_rouge(rag_outputs, references)

llm_bleu = compute_bleu(llm_outputs, references)
llm_rouge = compute_rouge(llm_outputs, references)

In [9]:
print("===== RAG =====")
print("BLEU:", rag_bleu)
print("ROUGE-L:", rag_rouge)

print("\n===== LLM =====")
print("BLEU:", llm_bleu)
print("ROUGE-L:", llm_rouge)

===== RAG =====
BLEU: 0.0029657536893201468
ROUGE-L: 0.04975314383818505

===== LLM =====
BLEU: 0.02468810519538035
ROUGE-L: 0.15799626716231463


In [10]:
from src.evaluation.metrics import compute_improvement

print("\n===== IMPROVEMENT =====")

print("BLEU improvement %:", compute_improvement(llm_bleu, rag_bleu))
print("ROUGE improvement %:", compute_improvement(llm_rouge, rag_rouge))


===== IMPROVEMENT =====
BLEU improvement %: -87.9871149857418
ROUGE improvement %: -68.50992448633482


In [11]:
import random

samples = random.sample(
    list(zip(questions, references, rag_outputs)),
    30
)

for q, ref, pred in samples:
    print("\nQUESTION:", q)
    print("\nGOLD:", ref)
    print("\nRAG:", pred)
    print("-"*80)


QUESTION: Are elevated homocysteine plasma levels related to peripheral arterial disease?

GOLD: Elevated HC is only slightly more related to PAD than to CAD and CVD. After adjustment for known risk factors, the effect size is small, and an association can no longer be observed between homocysteine and CAD and CVD.

RAG: Hypothetical! Why should I be on plasma?
text_chunk
--------------------------------------------------------------------------------

QUESTION: Does perioperative glucocorticosteroid treatment correlate with disturbance in surgical wound healing after treatment of facial fractures?

GOLD: With regard to DSWH, patients undergoing operative treatment of facial fractures can safely be administered doses of 30 mg or less of perioperative glucocorticosteroids equivalent to dexamethasone.

RAG: 2 (2)
Question: Inflammatory Morbidity due to Compou...
text_chunk     Question: Inflammatory Morbidity due to Compou...
text_chunk             
Question: Inflammatory Morbidity due 

In [ ]:
for q, a in samples:
    print("\nQUESTION:", q)
    print("ANSWER:", a)
    print("-" * 80)

In [ ]:
hallucination_rate = num_hallucinated / 30
print(hallucination_rate)

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

doc = SimpleDocTemplate("evaluation_report.pdf")
styles = getSampleStyleSheet()

content = []

content.append(Paragraph("RAG Evaluation Report", styles["Title"]))

content.append(Paragraph(f"RAG BLEU: {np.mean(rag_bleu)}", styles["Normal"]))
content.append(Paragraph(f"LLM BLEU: {np.mean(llm_bleu)}", styles["Normal"]))

content.append(Paragraph(f"RAG ROUGE-L: {np.mean(rag_rouge)}", styles["Normal"]))
content.append(Paragraph(f"LLM ROUGE-L: {np.mean(llm_rouge)}", styles["Normal"]))

doc.build(content)

In [ ]:
.